In [1]:
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login 

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

d:\apps\anaconda\envs\rl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Gymnasium and how it works

The library contains our environments. We'll use Gymnasium a lot in Deep RL.  
The Gymnasium library provides two things:  
- An interface that allows you to create RL environments.  
- A collection of environments (gym-control, atari, box2d).

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="The RL process" width="100%">

At each step:
- Our Agent receives a **state ($S_{0}$)** from the **Environment** — we receive the first frame of our game (Environment).
- Based on that **state ($S_{0}$),** the Agent takes an **action ($A_{0}$)** — our Agent will move to the right.
- The environment transitions to a **new** **state ($S_{1}$)** — new frame.
- The environment gives some **reward ($R_{1}$)** to the Agent — we’re not dead *(Positive Reward +1)*.


With Gymnasium:

1. We create our environment using `gymnasium.make(<environment_name>)`

2. We reset the environment to its initial state with `observation, info = env.reset()`

At each step:

3. Get an action using our model (in our example we take a random action)

4. Using `env.step(action)`, we perform this action in the environment and get
- `observation`: The new state (st+1)
- `reward`: The reward we get after executing the action
- `terminated`: Indicates if the episode terminated (agent reach the terminal state)
- `truncated`: Introduced with this new version, it indicates a timelimit or if an agent go out of bounds of the environment for instance.
- `info`: A dictionary that provides additional information (depends on the environment).

For more explanations: https://gymnasium.farama.org/api/env/#gymnasium.Env.step

If the episode is terminated:
- We reset the environment to its initial state with `observation = env.reset()`  

To take a random action:
- action = env.action_space.sample() 


In [ ]:
import gymnasium as gym

In [ ]:
env = gym.make('LunarLander-v3')

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

In [ ]:
observation, info = env.reset()

In [ ]:
for _ in range(20):
    action = env.action_space.sample()
    print(f'Action taken: {action}')

    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        print('Environment is reset')
        observation, info = env.reset()

env.close()

Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 0
Action taken: 3
Action taken: 3
Action taken: 1
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 0
Action taken: 2
Action taken: 2
Action taken: 1
Action taken: 0
Action taken: 0
Action taken: 3
Action taken: 2
Action taken: 1
Action taken: 2


### Create the LunarLander environment and understand how it works

In [ ]:
env = gym.make('LunarLander-v3')
observation, info = env.reset()
print('Observation space shape', env.observation_space.shape)
print('Sample observation', env.observation_space.sample)

Observation space shape (8,)
Sample observation <bound method Box.sample of Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)>


In [ ]:
print('Action space shape', env.action_space.n)
print('Action space sample', env.action_space.sample())

Action space shape 4
Action space sample 3


### Vectorized environment 
A method for stacking multiple independent environments into a single environment - this way, we'll have more diverse experiences during the training phase.

In [ ]:
env = make_vec_env('LunarLander-v3', n_envs=16)

In [ ]:
model = PPO(
    policy = 'MlpPolicy', 
    env=env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1
)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
model.learn(total_timesteps=1e6)
model_name = 'ppo-LunarLander-v3'
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 92.5     |
|    ep_rew_mean     | -188     |
| time/              |          |
|    fps             | 3677     |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 16384    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 91.7         |
|    ep_rew_mean          | -159         |
| time/                   |              |
|    fps                  | 2349         |
|    iterations           | 2            |
|    time_elapsed         | 13           |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0058881836 |
|    clip_fraction        | 0.0423       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | -3.97e-05    |
|    learning_r

In [ ]:
eval_env = Monitor(gym.make('LunarLander-v3', render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f'Mean reward={mean_reward:.2f} +/- {std_reward}')

Mean reward=264.59 +/- 17.297802215705374
